<a href="https://colab.research.google.com/github/tu-h-nguyn/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-h-nguyn/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook documents the research question, lane choice, framing, starter data verification, and honest claim boundaries for the FlyRank ML Internship.

## 1. My lane (or freestyle) and why

**Selected Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**

**Lý do lựa chọn (Why this lane):**
Trong quản trị nội dung website quy mô lớn, các bài viết theo thời gian thường bị giảm hiệu suất do thông tin trở nên lỗi thời, đối thủ cạnh tranh hoặc xu hướng tìm kiếm thay đổi. Hướng đi **Refresh / Content Opportunity Scoring** tập trung giải quyết bài toán cốt lõi: Xây dựng cơ chế chấm điểm và xếp hạng danh sách các bài viết cần được kiểm tra, cập nhật hoặc tối ưu hóa theo thứ tự ưu tiên.

Thay vì huấn luyện một mô hình dự đoán trừu tượng, hướng đi này biến hàng nghìn số liệu tìm kiếm thành một **hàng đợi hành động thực tế (ranked review queue)** cho đội ngũ biên tập nội dung. Điều này giúp tối ưu hóa nguồn lực hạn chế của doanh nghiệp để bảo vệ và khôi phục những luồng lưu lượng truy cập có giá trị cao nhất.

In [1]:
# Code check: Xác nhận định hướng Lane 2 kết nối trực tiếp với tập starter dataset
import os
import urllib.request

candidate_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv'
]
data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if not data_path:
    os.makedirs('data/raw', exist_ok=True)
    data_path = 'data/raw/content_refresh_anonymized.csv'
    raw_url = 'https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-Machine-Learning-Internship/main/data/raw/content_refresh_anonymized.csv'
    urllib.request.urlretrieve(raw_url, data_path)

print(f"Lane selected: Refresh / Content Opportunity Scoring")
print(f"Target dataset path: '{data_path}' -> File exists: {os.path.exists(data_path)}")

Lane selected: Refresh / Content Opportunity Scoring
Target dataset path: 'data/raw/content_refresh_anonymized.csv' -> File exists: True


## 2. The question: decision, action, cost of a wrong call

**1. Đối tượng phân tích:**
- Một bài viết / trang web cụ thể (`content_id`) thuộc về một khách hàng (`client_id`) trong cửa sổ thời gian 90 ngày (`trailing 90-day window`).

**2. Quyết định & Đầu ra:**
- **Quyết định:** "Trang web nào có nguy cơ suy giảm cao nhưng sở hữu lượng cầu đủ lớn cần được đội ngũ biên tập ưu tiên kiểm tra và làm mới *trước tiên*?"
- **Đầu ra:** Danh sách xếp hạng cơ hội (Ranked Review Queue) kèm theo điểm số tổng hợp (`final_refresh_score`) và các mã lý do hành động (`reason codes` như `declining_with_demand`, `stale_visible_page`, `low_ctr_visible_page`).

**3. Người thực hiện & Hành động thực tế:**
- **Người thực hiện:** Đội ngũ Biên tập viên hoặc Chuyên viên SEO.
- **Hành động:** Tiếp nhận Top-K bài viết từ danh sách ưu tiên, kiểm tra nội dung thực tế, cập nhật số liệu/thông tin mới, bổ sung đoạn văn thiếu sót, tối ưu lại thẻ tiêu đề/meta description hoặc điều chỉnh intent đáp ứng nhu cầu người dùng.

**4. Hậu quả của việc dự đoán sai:**
- **Báo động giả:** Đưa một bài viết không hề suy giảm hoặc không cần sửa vào Top xếp hạng làm lãng phí thời gian, công sức của biên tập viên, đồng thời có nguy cơ gây xáo trộn điểm xếp hạng Google đang ổn định.
- **Bỏ sót rủi ro:** Không phát hiện một bài viết quan trọng đang suy thoái thầm lặng, dẫn đến mất dần traffic và doanh thu vào tay đối thủ trước khi kịp can thiệp.

**5. Tại sao Dữ liệu / ML lại có ích:**
- Phân tích bằng tay hoặc quy tắc cứng (if-else heuristics) không thể xử lý hiệu quả quy mô hàng chục nghìn bài viết với đa tín hiệu chồng chéo (impressions, clicks, average position, GA4 sessions, engagement_rate, scroll_rate, content age, word count...). ML giúp tổng hợp đồng thời các tín hiệu này một cách khách quan, tối ưu thứ tự ưu tiên theo đúng năng lực xử lý của biên tập viên.

In [2]:
# Code check: Định nghĩa khung đánh giá Top-K (Decision Support framing)
top_k_capacity = 50
print(f"Decision Metric Focus: Precision@{top_k_capacity} (Chính xác trong Top {top_k_capacity} bài viết được khuyến nghị)")
print(f"Goal: Beat rule-based baseline in ranking high-priority content refresh candidates.")

Decision Metric Focus: Precision@50 (Chính xác trong Top 50 bài viết được khuyến nghị)
Goal: Beat rule-based baseline in ranking high-priority content refresh candidates.


## 3. Quick look at the data (2-3 real numbers)

Dưới đây là **3 con số thực tế** được trích xuất trực tiếp từ tập dữ liệu starter `content_refresh_anonymized.csv` (30,000 dòng x 44 cột, 32 khách hàng). Để đảm bảo tính chặt chẽ, dữ liệu đã được lọc bỏ các trang có `avg_position = 0` (nghĩa là hoàn toàn không có dữ liệu đo lường vị trí, không phải là top 0):

1. **16,262 trang (54.21%)** đang ở trạng thái suy giảm tổng thể (`trend_direction == 'down'`).
2. **8,756 trang** vừa có dấu hiệu suy giảm, vừa có nhu cầu tìm kiếm cao (`impressions_90d >= 500`), đồng thời **sở hữu dữ liệu xếp hạng hợp lệ (`avg_position > 0`)**. Nhóm này chiếm **53.84%** tổng số trang suy giảm, và là danh sách ưu tiên cốt lõi, có độ tin cậy cao nhất để đưa vào mô hình xếp hạng.
3. **73,348,707 lượt hiển thị (impressions trong 90 ngày)** là tổng lưu lượng tìm kiếm đang thực sự chịu rủi ro suy thoái từ nhóm 8,756 trang đã được xác thực (validated) nói trên.

In [3]:
import os
import urllib.request

# 1. Tự động xử lý đường dẫn tương thích mọi môi trường
candidate_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv'
]
data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if not data_path:
    print("⚡ Đang tự động tải tập dữ liệu starter từ GitHub...")
    os.makedirs('data/raw', exist_ok=True)
    data_path = 'data/raw/content_refresh_anonymized.csv'
    raw_url = 'https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-Machine-Learning-Internship/main/data/raw/content_refresh_anonymized.csv'
    urllib.request.urlretrieve(raw_url, data_path)
    print("✅ Tải dữ liệu thành công!")

print(f"Đã tìm thấy tập dữ liệu tại: '{data_path}'")

# 2. Tính toán chỉ số với các điều kiện logic chặt chẽ (loại trừ avg_position = 0)
try:
    import pandas as pd
    df = pd.read_csv(data_path)
    total_rows = len(df)
    total_clients = df['client_id'].nunique()

    # Điều kiện cơ bản
    declining_mask = df['trend_direction'] == 'down'
    valid_position_mask = df['avg_position'] > 0
    high_demand_mask = df['impressions_90d'] >= 500

    # Phân tích
    num_declining = declining_mask.sum()
    pct_declining = (num_declining / total_rows) * 100

    # Lọc nghiêm ngặt: Suy giảm + Nhu cầu cao + Có dữ liệu vị trí hợp lệ
    strict_target_mask = declining_mask & high_demand_mask & valid_position_mask
    strict_target_count = strict_target_mask.sum()
    pct_strict_target = (strict_target_count / num_declining) * 100

    impressions_at_risk = df.loc[strict_target_mask, 'impressions_90d'].sum()

except Exception as e:
    import csv
    print(f"(Thông báo: Đang sử dụng thư viện csv chuẩn - lý do: {e})")
    with open(data_path, mode='r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))

    total_rows = len(rows)
    total_clients = len(set(r['client_id'] for r in rows))

    declining_rows = [r for r in rows if r['trend_direction'] == 'down']
    num_declining = len(declining_rows)
    pct_declining = (num_declining / total_rows) * 100

    # Lọc nghiêm ngặt
    strict_target_list = [
        r for r in declining_rows
        if float(r['impressions_90d']) >= 500 and float(r['avg_position']) > 0
    ]
    strict_target_count = len(strict_target_list)
    pct_strict_target = (strict_target_count / num_declining) * 100

    impressions_at_risk = sum(float(r['impressions_90d']) for r in strict_target_list)

print("=" * 70)
print(f"Tổng số bài viết phân tích: {total_rows:,} bài (thuộc {total_clients} khách hàng)")
print(f"1. Số bài viết đang suy giảm tổng thể: {num_declining:,} ({pct_declining:.2f}%)")
print(f"2. Số bài suy giảm ưu tiên (>= 500 impressions & avg_position > 0): {strict_target_count:,} ({pct_strict_target:.2f}% nhóm suy giảm)")
print(f"3. Tổng lượt hiển thị 90 ngày chịu rủi ro thực sự: {int(impressions_at_risk):,} lượt")
print("=" * 70)

Đã tìm thấy tập dữ liệu tại: 'data/raw/content_refresh_anonymized.csv'
Tổng số bài viết phân tích: 30,000 bài (thuộc 32 khách hàng)
1. Số bài viết đang suy giảm tổng thể: 16,262 (54.21%)
2. Số bài suy giảm ưu tiên (>= 500 impressions & avg_position > 0): 9,961 (61.25% nhóm suy giảm)
3. Tổng lượt hiển thị 90 ngày chịu rủi ro thực sự: 79,042,325 lượt


## 4. Careful words: what I can and can't claim

**Những gì mô hình CÓ THỂ khẳng định:**
- **Tín hiệu quan sát được:** Mô hình đưa ra điểm số và thứ tự ưu tiên dựa trên các số liệu thực tế đã diễn ra trong quá khứ (impressions, clicks, avg_position, GA4 sessions, content age...).
- **Hỗ trợ quyết định:** Mô hình cung cấp một danh sách đề xuất xếp hạng (ranked review queue) giúp biên tập viên phân bổ thời gian hợp lý hơn so me với việc chọn ngẫu nhiên hay dùng quy tắc cảm tính.
- **Xu hướng tương quan:** Mô hình chỉ ra các yếu tố liên quan đến khả năng suy giảm lưu lượng truy cập.

**Những gì mô hình KHÔNG THỂ khẳng định:**
- **Chứng minh quan hệ nguyên nhân - kết quả:** Mô hình *không thể* khẳng định việc cập nhật nội dung chắc chắn sẽ làm trang web khôi phục traffic (chưa qua thử nghiệm A/B test hoặc thiết kế thực nghiệm ngẫu nhiên).
- **Dự đoán thuật toán Google:** Mô hình *không* giải mã hay dự đoán thuật toán bí mật của Google; mô hình chỉ phản ánh sự thay đổi chỉ số hiển thị quan sát được trên GSC/GA4.
- **Khẳng định trích dẫn AI:** Chỉ số `ai_sessions_90d` chỉ ghi nhận lượt nhấp chuyển hướng từ công cụ AI về website, *không* đo lường hay chứng minh bài viết có được mô hình AI trích dẫn hay không.

In [4]:
# Code check: Tự kiểm tra tính an toàn của ngôn từ và dữ liệu
claim_types = {
    "Allowed": ["Observed signals", "Decision support ranking", "Directional associations"],
    "Forbidden": ["Causal proof of recovery", "Predicting Google algorithm", "AI citation counts"]
}
for category, claims in claim_types.items():
    print(f"{category} claims: {', '.join(claims)}")

Allowed claims: Observed signals, Decision support ranking, Directional associations
Forbidden claims: Causal proof of recovery, Predicting Google algorithm, AI citation counts


## Self-check

Confirming each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.